# Oracle lower bound of the SLAE-ERR MAE, for a known noise level

Assumes independent Gaussian experimental noise: observed error e = m + n, with model error m and noise n ~ N(0, sigma^2).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize, stats

PRED_PATH = Path("../../data/analysis/Fig5/predictions_N_ensemble_10seed_sorted_centered_err.json")
FIG = Path("Figs")

# ---- CONFIG ----
SIGMAS = (0.5,)                        # SD of the noise of one measurement (ppm); add values to compare
N_BOOT = 2000                           # protein-cluster bootstrap
SEED = 0
TAG = ""                                # suffix of the output file

## Lowest MAE when m is known

g(r) = lowest MAE / sigma for r = |m| / sigma. The best prediction of |m + n| is its median.

In [ ]:
Z = np.linspace(-9, 9, 36001)              # standard normal noise, on a fine grid
WEIGHT = stats.norm.pdf(Z) * (Z[1] - Z[0])


def folded_median(r):
    """Median of |r + Z|: solves Phi(c - r) - Phi(-c - r) = 1/2."""
    return optimize.brentq(lambda c: stats.norm.cdf(c - r) - stats.norm.cdf(-c - r) - 0.5, 0, r + 10)


def g(r):
    """E| |r + Z| - median |, by numerical integration."""
    return float((np.abs(np.abs(r + Z) - folded_median(r)) * WEIGHT).sum())


R_GRID = np.linspace(0, 8, 801)             # beyond r = 8, g is constant
G_GRID = np.array([g(r) for r in R_GRID])

# checks against the closed forms and a simulation
c = stats.norm.ppf(0.75)
assert abs(G_GRID[0] - (4 * stats.norm.pdf(c) - 2 * stats.norm.pdf(0))) < 1e-4, G_GRID[0]    # m = 0
assert abs(G_GRID[-1] - np.sqrt(2 / np.pi)) < 1e-4, G_GRID[-1]                               # |m| >> sigma
assert G_GRID.min() >= G_GRID[0] - 1e-4 and G_GRID.max() <= np.sqrt(2 / np.pi) + 1e-4       # stays within the two limits
rng = np.random.default_rng(1)
x = np.abs(1.0 + rng.normal(size=4_000_000))
assert abs(np.abs(x - np.median(x)).mean() - g(1.0)) < 2e-3
print(f"g(0) = {G_GRID[0]:.4f}, g(1) = {g(1.0):.4f}, g(8) = {G_GRID[-1]:.4f}, max g = {G_GRID.max():.4f}")


def oracle_coefficient(abs_m, sigma):
    """g(|m| / sigma) of each residue."""
    return np.interp(abs_m / sigma, R_GRID, G_GRID)

## Load residues

In [ ]:
raw = json.load(open(PRED_PATH))
rows = []
for entry in raw["pred"]:
    valid = np.array(raw["mask"][entry], bool) & np.array(raw["oof_mask"][entry], bool)
    rows.append(pd.DataFrame(dict(entry=entry, pred=np.array(raw["pred"][entry], float)[valid],
                                  obs=np.array(raw["oof_resid"][entry], float)[valid])))
df = pd.concat(rows, ignore_index=True)
assert np.isfinite(df[["pred", "obs"]].to_numpy()).all() and (df.obs >= 0).all()
print(f"{len(df)} residues, {df.entry.nunique()} proteins")

## Bounds

In [ ]:
def bounds(d, sigma):
    """MAE of SLAE-ERR and its lower bound for this sigma."""
    obs = d.obs.to_numpy()
    return dict(slae_err_mae=np.abs(d.pred.to_numpy() - obs).mean(),
                # m is not observed: |m| = |e|, or |e| with the noise variance removed
                slae_err_oracle_e=sigma * oracle_coefficient(obs, sigma).mean(),
                slae_err_oracle_shrunk=sigma * oracle_coefficient(np.sqrt(np.maximum(obs ** 2 - sigma ** 2, 0)), sigma).mean())


parts = {e: g_ for e, g_ in df.groupby("entry")}
entries = np.array(list(parts))
rng = np.random.default_rng(SEED)
picks = [rng.choice(entries, len(entries), replace=True) for _ in range(N_BOOT)]
rows = []
for sigma in SIGMAS:
    point = bounds(df, sigma)
    boot = pd.DataFrame(bounds(pd.concat([parts[e] for e in pick], ignore_index=True), sigma) for pick in picks)
    row = dict(sigma=sigma, **point)
    for col in point:
        row[f"{col}_lo"], row[f"{col}_hi"] = np.percentile(boot[col], [2.5, 97.5])
    assert 0.4732 * sigma - 1e-3 <= point["slae_err_oracle_shrunk"] <= point["slae_err_oracle_e"] <= 0.7979 * sigma + 1e-3
    rows.append(row)
T = pd.DataFrame(rows)

FIG.mkdir(exist_ok=True)
path = FIG / f"T_oracle_mae{TAG}.csv"
T.to_csv(path, index=False)
assert len(pd.read_csv(path)) == len(SIGMAS)
print("saved", path.name)
T[["sigma", "slae_err_mae", "slae_err_mae_lo", "slae_err_mae_hi", "slae_err_oracle_shrunk", "slae_err_oracle_e", "slae_err_oracle_shrunk_lo", "slae_err_oracle_shrunk_hi", "slae_err_oracle_e_lo", "slae_err_oracle_e_hi"]].round(3)